In [1]:
import logging
import csv

import os
import sys
sys.path.append("../..")
from pathlib import Path
from tqdm import tqdm

from monai.networks.nets import DiffusionModelUNet
import torch
from monai.config import print_config
from monai.utils import set_determinism
from monai.data import CacheDataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
import monai.transforms as transforms

import utils.custom_transforms as custom_transforms
import utils.simplex_ddpm as simplex_ddpm

import AnoDDPM.simplex as simplex

import numpy as np
import matplotlib.pyplot as plt
import copy


In [2]:
DEVICE_TYPE = "cuda:0"
device = torch.device(DEVICE_TYPE)

set_determinism(0)

ROOT_DIR = "/home/fehrdelt/bettik/"
#ROOT_DIR = "/bettik/PROJECTS/pr-gin5_aini/fehrdelt/"

In [3]:

IMAGE_SIZE = 128

torch.backends.cudnn.benchmark = True
torch.set_num_threads(torch.get_num_threads())
torch.autograd.set_detect_anomaly(False)

In [8]:

batch_size = 10
num_workers = 8


# transforms
test_transforms_adc = transforms.Compose(
    [
        transforms.LoadImage(image_only=True),
        transforms.EnsureChannelFirst(),
        transforms.ResizeWithPadOrCrop(spatial_size=(IMAGE_SIZE, IMAGE_SIZE, IMAGE_SIZE)),
        custom_transforms.ScaleIntensityFromHistogramPeak(target_value=1000.0),
        transforms.ScaleIntensityRange(a_min=0.0, a_max=3000.0, b_min=0.0, b_max=1.0, clip=True),
        custom_transforms.SetBackgroundToZero(),
        #transforms.EnsureType(device=device, track_meta=False)
    ]
)

test_transforms_flair = transforms.Compose(
    [
        transforms.LoadImage(image_only=True),
        transforms.EnsureChannelFirst(),
        transforms.ResizeWithPadOrCrop(spatial_size=(IMAGE_SIZE, IMAGE_SIZE, IMAGE_SIZE)),
        custom_transforms.ScaleIntensityFromHistogramPeak(target_value=200.0),
        transforms.ScaleIntensityRange(a_min=0.0, a_max=450.0, b_min=0.0, b_max=1.0, clip=True),
        custom_transforms.SetBackgroundToZero(),
        #transforms.EnsureType(device=device, track_meta=False)
    ]
)


In [20]:
val_csv = os.path.join(ROOT_DIR, f"AnoDiffExperiments/data_splits_lists/final_flair_dataset_small/val.csv")
val_images_path = []

with open(val_csv, mode='r') as file:
    reader = csv.reader(file)
    for line in tqdm(reader):

        val_images_path.append(ROOT_DIR+line[0])

val_datalist = val_images_path

val_ds = CacheDataset(data=val_datalist, transform=test_transforms_flair)

val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True
)

In [22]:
large_group = ['sub-strokecase0023_ses-0001_msk.nii.gz', 'sub-strokecase0031_ses-0001_msk.nii.gz', 'sub-strokecase0047_ses-0001_msk.nii.gz', 'sub-strokecase0048_ses-0001_msk.nii.gz', 'sub-strokecase0062_ses-0001_msk.nii.gz', 'sub-strokecase0066_ses-0001_msk.nii.gz', 'sub-strokecase0081_ses-0001_msk.nii.gz', 'sub-strokecase0083_ses-0001_msk.nii.gz', 'sub-strokecase0087_ses-0001_msk.nii.gz', 'sub-strokecase0091_ses-0001_msk.nii.gz', 'sub-strokecase0123_ses-0001_msk.nii.gz', 'sub-strokecase0161_ses-0001_msk.nii.gz', 'sub-strokecase0162_ses-0001_msk.nii.gz', 'sub-strokecase0171_ses-0001_msk.nii.gz', 'sub-strokecase0176_ses-0001_msk.nii.gz', 'sub-strokecase0201_ses-0001_msk.nii.gz', 'sub-strokecase0211_ses-0001_msk.nii.gz', 'sub-strokecase0222_ses-0001_msk.nii.gz', 'sub-strokecase0223_ses-0001_msk.nii.gz', 'sub-strokecase0023_ses-0001_msk.nii.gz', 'sub-strokecase0031_ses-0001_msk.nii.gz', 'sub-strokecase0047_ses-0001_msk.nii.gz', 'sub-strokecase0048_ses-0001_msk.nii.gz', 'sub-strokecase0062_ses-0001_msk.nii.gz', 'sub-strokecase0066_ses-0001_msk.nii.gz', 'sub-strokecase0081_ses-0001_msk.nii.gz', 'sub-strokecase0083_ses-0001_msk.nii.gz', 'sub-strokecase0087_ses-0001_msk.nii.gz', 'sub-strokecase0091_ses-0001_msk.nii.gz', 'sub-strokecase0123_ses-0001_msk.nii.gz', 'sub-strokecase0161_ses-0001_msk.nii.gz', 'sub-strokecase0162_ses-0001_msk.nii.gz', 'sub-strokecase0171_ses-0001_msk.nii.gz', 'sub-strokecase0176_ses-0001_msk.nii.gz', 'sub-strokecase0201_ses-0001_msk.nii.gz', 'sub-strokecase0211_ses-0001_msk.nii.gz', 'sub-strokecase0222_ses-0001_msk.nii.gz', 'sub-strokecase0223_ses-0001_msk.nii.gz', 'sub-strokecase0230_ses-0001_msk.nii.gz', 'sub-strokecase0237_ses-0001_msk.nii.gz', 'sub-strokecase0240_ses-0001_msk.nii.gz', 'sub-strokecase0246_ses-0001_msk.nii.gz']
large_group_images_adc = [ROOT_DIR+"datasets/final_adc_dataset_small/ISLES_registered/"+filename.replace("msk", "adc") for filename in large_group]
large_group_images_flair = [ROOT_DIR+"datasets/final_flair_dataset_small/isles_registered/"+filename.replace("msk", "FLAIR") for filename in large_group]



test_anomaly_large_ds_adc = CacheDataset(data=large_group_images_adc, transform=test_transforms_adc)
test_anomaly_large_ds_flair = CacheDataset(data=large_group_images_flair, transform=test_transforms_flair)

test_anomaly_large_loader_adc = DataLoader(
        test_anomaly_large_ds_adc, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True
)

test_anomaly_large_loader_flair = DataLoader(
        test_anomaly_large_ds_flair, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True
)

In [23]:
# Plot middle axial, coronal, and sagittal slices of first volumes

#first_batch = next(iter(test_anomaly_large_loader_flair))  # shape: (B, 1, H, W, D)
#first_batch = next(iter(val_loader))  # shape: (B, 1, H, W, D)
first_batch = next(iter(test_anomaly_large_loader_adc))  # shape: (B, 1, H, W, D)

num_to_show = min(8, first_batch.shape[0])

fig, axes = plt.subplots(num_to_show, 3, figsize=(9, 3 * num_to_show))
if num_to_show == 1:
    axes = axes.reshape(1, 3)

for i in range(num_to_show):
    vol = first_batch[i, 0]  # (H, W, D)

    print(vol.shape)

    H, W, D = vol.shape
    axial_idx = D // 2
    coronal_idx = W // 2
    sagittal_idx = H // 2

    vol = vol[...,  axial_idx-42:axial_idx+30]  # Crop to IMAGE_SIZE in D dimension

    # Axial (H x W at middle D)
    axes[i, 0].imshow(vol[:, :, vol.shape[2] // 2], cmap="gray")
    axes[i, 0].set_title("Axial" if i == 0 else "")
    axes[i, 0].axis("off")

    # Coronal (H x D at middle W)
    axes[i, 1].imshow(vol[:, coronal_idx, :], cmap="gray")
    axes[i, 1].set_title("Coronal" if i == 0 else "")
    axes[i, 1].axis("off")

    # Sagittal (W x D at middle H)
    axes[i, 2].imshow(vol[sagittal_idx, :, :], cmap="gray")
    axes[i, 2].set_title("Sagittal" if i == 0 else "")
    axes[i, 2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
model = DiffusionModelUNet(
    spatial_dims=2,
    in_channels=1,
    out_channels=1,
    channels=(32, 64, 64, 64),
    attention_levels=(False, True, True, True),
    num_head_channels=8
)
model.to(device)

model_path = ROOT_DIR+"AnoDiffExperiments/experiment_0/exp_0_7/models/exp_0_7_best_model.pth"
model.load_state_dict(torch.load(model_path, map_location=DEVICE_TYPE))

model.eval()

